# 영업 메일 파이프라인 — LangChain 버전

`openai_agent_code.ipynb`와 **똑같은 일**을 LangChain으로 만듭니다.
같은 문제를 다른 도구로 풀어 보면 각 프레임워크가 무엇을 대신해 주는지 분명해집니다.

## 시나리오 (동일)

**주식회사 윈스테크넷**의 **스나이퍼 APT**를 파는 Sales Group.
메일 스타일이 다른 판매원 세 명이 초안을 쓰고, 그중 하나를 골라 보냅니다.

| | 성향 |
|---|---|
| 판매원1 | 매우 정직 |
| 판매원2 | 유머러스 |
| 판매원3 | 매우 공격적 ("안 사면 망한다") |

## 먼저 알아야 할 것 — LangChain은 "에이전트 프레임워크"가 아닙니다

이름 때문에 오해하기 쉽습니다. LangChain의 본체는 **연결(chain)** 입니다.

```python
프롬프트 | 모델 | 출력파서
```

파이프 기호 `|`로 부품을 이어 붙입니다. 리눅스 셸의 파이프와 같은 발상입니다.
이것을 **LCEL**(LangChain Expression Language)이라고 부릅니다.

| | OpenAI Agents SDK | LangChain |
|---|---|---|
| 기본 단위 | `Agent` (성격을 가진 주체) | `Runnable` (입력→출력 부품) |
| 엮는 법 | 도구·핸드오프 | `|` 로 연결 |
| 도구 실행 | **SDK가 대신 해준다** | **우리가 직접 한다** |
| 반복 판단 | SDK가 알아서 돌린다 | 우리가 루프를 짠다 |

마지막 두 줄이 이 노트북의 핵심입니다. Step 7에서 직접 확인합니다.

> **실행 순서**: 위에서부터 차례대로 실행하세요. `agent/` 폴더에서 실행하는 것을 전제로 합니다.

## Step 0. 준비

LangChain은 패키지가 잘게 나뉘어 있습니다. 어떤 것을 어디서 가져오는지 먼저 봅니다.

| 패키지 | 무엇이 들어 있나 |
|---|---|
| `langchain_openai` | OpenAI 연결 (`ChatOpenAI`) |
| `langchain_core.prompts` | 프롬프트 틀 |
| `langchain_core.output_parsers` | 모델 출력 다듬기 |
| `langchain_core.runnables` | 부품 조립 도구 |
| `langchain_core.tools` | 도구 만들기 |

모델을 바꿔 끼우기 쉽게 하려고 이렇게 나눠 두었습니다.
`ChatOpenAI`를 `ChatAnthropic`으로 바꾸면 나머지 코드는 그대로 돌아갑니다.

In [1]:
import os
import smtplib
from email.message import EmailMessage

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnableLambda
from langchain_core.tools import tool

load_dotenv(override=True)

MODEL_NAME = "gpt-5.4-mini"
llm = ChatOpenAI(model=MODEL_NAME)

print("모델:", MODEL_NAME)
print("LangSmith 추적:", os.getenv("LANGSMITH_TRACING"))
print("LangSmith 프로젝트:", os.getenv("LANGSMITH_PROJECT"))

모델: gpt-5.4-mini
LangSmith 추적: true
LangSmith 프로젝트: secuirty_tracer


> **추적은 자동입니다.** OpenAI Agents SDK에서는 `with trace("이름")`으로 직접 묶었지만,
> LangChain은 `.env`에 `LANGSMITH_TRACING=true`만 있으면 **모든 실행이 자동으로 기록**됩니다.
> <https://smith.langchain.com> 에서 위 프로젝트 이름으로 찾을 수 있습니다.
>
> 편하지만 단점도 있습니다. **끄는 것을 잊으면 모든 프롬프트가 외부로 전송됩니다.**
> 사내 문서나 고객 정보를 다룰 때는 이 설정을 반드시 확인하세요.

## Step 1. 메일 보내기 준비 (안전장치 포함)

`openai_agent_code.ipynb`와 같습니다. 프레임워크와 무관한 부분이라 그대로 씁니다.

```python
SEND_FOR_REAL = False   # 기본값: 보내지 않고 화면에만 보여준다
```

실습 중 셀을 다시 실행할 때마다 메일이 나가면 곤란합니다.
**되돌릴 수 없는 동작은 기본을 꺼 두세요.**

In [2]:
EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS")
EMAIL_SMTP_SERVER = os.getenv("EMAIL_SMTP_SERVER")
EMAIL_APP_PASSWORD = os.getenv("EMAIL_APP_PASSWORD")

SEND_FOR_REAL = False   # True로 바꾸면 진짜 메일이 나갑니다

HAS_EMAIL_CONFIG = all([EMAIL_ADDRESS, EMAIL_SMTP_SERVER, EMAIL_APP_PASSWORD])


def send_email(subject, text_body):
    """SMTP로 실제 메일을 보낸다."""
    msg = EmailMessage()
    msg["From"] = EMAIL_ADDRESS
    msg["To"] = EMAIL_ADDRESS
    msg["Subject"] = subject
    msg.set_content(text_body)
    with smtplib.SMTP(EMAIL_SMTP_SERVER, 587) as server:
        server.starttls()
        server.login(EMAIL_ADDRESS, EMAIL_APP_PASSWORD)
        server.send_message(msg)


def send_message(subject, text_body):
    """스위치를 보고 실제로 보낼지 화면에 보여줄지 정한다."""
    if SEND_FOR_REAL and HAS_EMAIL_CONFIG:
        send_email(subject, text_body)
        return "메일을 실제로 보냈습니다."
    print("=" * 66)
    print("[미리보기] 실제로 보내지 않았습니다")
    print("제목:", subject)
    print("-" * 66)
    print(text_body)
    print("=" * 66)
    return "미리보기로 출력했습니다."


print("발송 모드:", "실제 발송" if SEND_FOR_REAL else "미리보기 (화면 출력만)")

발송 모드: 미리보기 (화면 출력만)


## Step 2. 판매원 세 명 = 체인 세 개

여기가 첫 번째 큰 차이입니다.

```python
# OpenAI Agents SDK
sales_agent1 = Agent(name="정직한 판매원", instructions=..., model=...)

# LangChain
chain1 = prompt1 | llm | StrOutputParser()
```

LangChain에는 "판매원"이라는 **개체가 없습니다.** 대신 **거쳐 가는 파이프라인**이 있습니다.

### 세 부품이 하는 일

| 부품 | 입력 | 출력 |
|---|---|---|
| `ChatPromptTemplate` | `{"제품": "..."}` 같은 딕셔너리 | 메시지 목록 |
| `llm` | 메시지 목록 | `AIMessage` 객체 |
| `StrOutputParser` | `AIMessage` 객체 | **문자열** |

`StrOutputParser`가 없으면 결과가 `AIMessage(content='...', ...)` 객체로 나옵니다.
다음 부품에 넘기거나 출력할 때 매번 `.content`를 붙여야 하니, 끝에 붙여서 문자열로 만듭니다.

In [3]:
intro = """당신은 주식회사 윈스테크넷의 영업 담당자입니다.
'스나이퍼 APT'라는 지능형 지속 위협(APT) 대응 솔루션을 판매합니다.
이 제품은 알려지지 않은 악성코드와 표적 공격을 네트워크 단에서 탐지하고 차단합니다.
당신의 일은 잠재 고객에게 보낼 영업 메일을 쓰는 것입니다.
메일은 한국어로 쓰고, 제목과 본문을 모두 포함하세요.
"""

style1 = """당신의 메일 스타일은 매우 정직합니다.
과장하거나 부풀리지 않습니다. 확인된 사실만 씁니다.
제품이 해결하지 못하는 부분이 있다면 그것도 솔직하게 밝힙니다."""

style2 = """당신의 메일 스타일은 유머러스합니다.
가볍고 재치 있는 표현으로 읽는 사람의 부담을 덜어줍니다.
다만 제품 정보 자체는 정확하게 전달합니다."""

style3 = """당신의 메일 스타일은 매우 공격적입니다.
이 제품을 도입하지 않으면 고객사가 큰 피해를 입고 사업이 무너질 수 있다는 점을 강하게 압박합니다.
위기감을 최대한 강조하고 즉시 결정할 것을 요구합니다."""

### 프롬프트 틀을 한 번만 만들기

`ChatPromptTemplate`에 `{style}`과 `{request}`라는 **빈칸**을 둡니다.
틀 하나를 세 판매원이 공유하고, 빈칸만 다르게 채웁니다.

`intro`를 f-string으로 세 번 이어 붙이는 대신 이렇게 하는 이유:
빈칸이 있는 틀은 **재사용할 수 있고**, 나중에 입력이 어디로 들어가는지 코드만 봐도 보입니다.

> **주의**: 프롬프트 틀 안의 중괄호 `{}`는 빈칸으로 해석됩니다.
> JSON 예시처럼 진짜 중괄호를 쓰려면 `{{`, `}}`로 두 번 써야 합니다.

In [4]:
prompt = ChatPromptTemplate.from_messages([
    ("system", intro + "\n{style}"),
    ("human", "{request}"),
])

# 같은 틀에 성향만 다르게 채운 체인 세 개.
parser = StrOutputParser()
chain1 = prompt.partial(style=style1) | llm | parser
chain2 = prompt.partial(style=style2) | llm | parser
chain3 = prompt.partial(style=style3) | llm | parser

print("체인 세 개 준비 완료")
print("chain1 구조:", " -> ".join(type(s).__name__ for s in chain1.steps))

체인 세 개 준비 완료
chain1 구조: ChatPromptTemplate -> ChatOpenAI -> StrOutputParser


`.partial(style=...)`은 **빈칸 하나를 미리 채워 둔** 새 체인을 만듭니다.
그래서 실행할 때는 나머지 빈칸(`request`)만 주면 됩니다.

출력에 보이는 `ChatPromptTemplate -> ChatOpenAI -> StrOutputParser`가
`|`로 이어 붙인 결과입니다. **체인은 부품 목록을 들고 있는 객체일 뿐**입니다.

## Step 3. 한 명 실행 — 스트리밍

`.invoke()`는 다 끝난 뒤 한 번에, `.stream()`은 만들어지는 대로 조금씩 줍니다.

OpenAI Agents SDK에서는 스트림에서 글자 조각만 골라내는 `if` 문이 필요했습니다.

```python
if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
```

LangChain은 **`StrOutputParser`가 끝에 있어서 조각이 이미 문자열**입니다. 그대로 출력하면 됩니다.
파서 하나가 스트리밍까지 정리해 주는 셈입니다.

In [5]:
for piece in chain1.stream({"request": "영업 메일 초안을 하나 써 주세요"}):
    print(piece, end="", flush=True)

제목: 지능형 위협 대응을 위한 네트워크 기반 탐지/차단 솔루션 안내

안녕하세요,  
주식회사 윈스테크넷 영업 담당자입니다.

최근 알려지지 않은 악성코드와 표적형 공격은 단말 백신만으로는 대응이 어려운 경우가 많습니다.  
이와 관련해 저희는 네트워크 단에서 지능형 지속 위협(APT) 징후를 탐지하고 차단하는 **‘스나이퍼 APT’**를 안내드리고자 연락드렸습니다.

스나이퍼 APT는 네트워크 트래픽을 분석해 알려지지 않은 악성코드나 표적 공격의 의심 징후를 탐지하고, 설정된 정책에 따라 차단할 수 있도록 지원합니다.  
즉, 내부 단말에서 발생한 이상 행위가 외부와 통신하는 과정에서 드러나는 경우 이를 네트워크 수준에서 확인하는 데 활용할 수 있습니다.

다만, 솔직히 말씀드리면 이 제품만으로 모든 보안 문제를 해결할 수는 없습니다.  
예를 들어 사용자 계정 탈취, 정상 프로그램을 악용한 공격, 이미 암호화된 트래픽 내부의 세부 내용 확인 등은 환경과 정책에 따라 한계가 있을 수 있습니다.  
따라서 단말 보안, 계정 관리, 로그 분석 등 다른 보안 체계와 함께 운영하는 것이 더 적절합니다.

원하시면 간단한 제품 소개 자료나, 현재 운영 환경에 맞는 적용 가능 범위를 함께 검토해 드리겠습니다.  
편하신 시간에 회신 주시면 감사하겠습니다.

감사합니다.  
주식회사 윈스테크넷  
영업 담당자 드림

## Step 4. 세 명을 동시에 — RunnableParallel

`asyncio.gather` 대신 **`RunnableParallel`** 을 씁니다.

```python
RunnableParallel(honest=chain1, humorous=chain2, aggressive=chain3)
```

이것도 하나의 `Runnable`입니다. 입력을 **셋 모두에게 그대로 전달**하고,
결과를 **딕셔너리로 모아** 돌려줍니다.

| | OpenAI Agents SDK | LangChain |
|---|---|---|
| 문법 | `await asyncio.gather(...)` | `parallel.invoke(...)` |
| `async` 필요? | 필요 | **불필요** (안에서 알아서 처리) |
| 결과 형태 | 리스트 (순서로 구분) | **딕셔너리** (이름으로 구분) |
| 다시 조립 | 별도 코드 필요 | 그대로 다음 부품에 `|` 로 연결 |

이름으로 구분되는 점이 실무에서 편합니다. 리스트 인덱스 `results[2]`가 누구였는지
나중에 보면 헷갈립니다.

In [6]:
writers = RunnableParallel(
    honest=chain1,
    humorous=chain2,
    aggressive=chain3,
)

drafts = writers.invoke({"request": "영업 메일 초안을 하나 써 주세요"})

LABELS = {"honest": "판매원1 (정직)",
          "humorous": "판매원2 (유머)",
          "aggressive": "판매원3 (공격)"}

for key, draft in drafts.items():
    print("=" * 66)
    print(LABELS[key])
    print("=" * 66)
    print(draft)
    print()

판매원1 (정직)
**제목: 네트워크 단에서 APT 위협 탐지·차단을 지원하는 ‘스나이퍼 APT’ 안내**

안녕하세요,  
주식회사 윈스테크넷입니다.

최근에는 알려진 악성코드뿐 아니라, 기존 시그니처 기반 탐지만으로는 확인이 어려운 표적 공격과 지능형 지속 위협(APT)에 대한 대응이 중요해지고 있습니다.  
저희 **‘스나이퍼 APT’**는 이런 위협을 **네트워크 단에서 탐지하고 차단**하는 데 초점을 둔 솔루션입니다.

스나이퍼 APT는 다음과 같은 상황에서 활용될 수 있습니다.

- 알려지지 않은 악성코드의 이상 징후 탐지
- 표적 공격과 APT 트래픽 식별
- 네트워크 구간에서의 차단 및 대응 지원

다만 솔직하게 말씀드리면,  
**스나이퍼 APT가 모든 보안 문제를 단독으로 해결하는 것은 아닙니다.**  
예를 들어 엔드포인트 자체의 취약점, 계정 탈취, 내부 운영 절차상의 문제까지 모두 대신해 주지는 않습니다. 따라서 기존 보안 장비, 보안 정책, 사용자 관리와 함께 운영하실 때 더 적절합니다.

관심이 있으시다면, 귀사의 환경에 맞춰  
- 적용 가능 여부  
- 운영 방식  
- 도입 시 확인해야 할 사항  
을 간단히 안내드리겠습니다.

검토해 주셔서 감사합니다.  
필요하시면 짧은 소개 자료 형태로도 다시 정리해 드리겠습니다.

감사합니다.  
주식회사 윈스테크넷 드림

판매원2 (유머)
**제목:** 보이지 않는 위협, 네트워크 앞에서 미리 잡아두는 방법 — 스나이퍼 APT 소개드립니다

안녕하세요,  
주식회사 윈스테크넷입니다.

요즘 보안 담당자분들께서는  
“정체를 알 수 없는 친구가 우리망에 슬그머니 들어오려는 것 같은데요?”  
이런 느낌, 한 번쯤 받아보셨을 텐데요.  
그럴 때 필요한 것이 바로 **스나이퍼 APT**입니다.

**스나이퍼 APT**는  
알려지지 않은 악성코드와 표적 공격을 **네트워크 단에서 탐지하고 차단하는 지능형 지속 위협(APT) 대응 솔루션**입니다.  
쉽게 말해, 이상한 기운이 포착되면 “잠

### 세 초안이 얼마나 다른가

성향이 결과에 정말 반영됐는지 **지시문만 보고 믿지 말고 숫자로 확인**합니다.

In [7]:
FEAR_WORDS = ["위험", "피해", "위협", "치명", "심각", "즉시", "지금", "손실", "마비", "붕괴"]

print(f"{'':16} {'글자수':>6} {'위기낱말':>7}")
for key, draft in drafts.items():
    hits = {w: draft.count(w) for w in FEAR_WORDS if w in draft}
    print(f"{LABELS[key]:16} {len(draft):>6} {sum(hits.values()):>7}   {hits}")

                    글자수    위기낱말
판매원1 (정직)           667       3   {'위협': 3}
판매원2 (유머)           669       4   {'위협': 4}
판매원3 (공격)          1003       9   {'위협': 4, '즉시': 1, '지금': 3, '마비': 1}


## Step 5. 고르는 체인

선택도 체인 하나입니다. 다만 입력이 **딕셔너리**로 들어오므로,
프롬프트에 넣기 전에 **하나의 글로 합쳐야** 합니다.

그 합치는 일을 하는 것이 `RunnableLambda`입니다.
평범한 파이썬 함수를 체인의 부품으로 만들어 줍니다.

> 여기가 LangChain의 성격을 잘 보여줍니다. 데이터를 다듬는 일까지
> **체인 안에 부품으로 넣습니다.** 체인 밖에서 따로 처리해도 되지만,
> 안에 넣으면 전체가 하나의 `Runnable`이 되어 그대로 재사용·추적됩니다.

In [8]:
def join_drafts(drafts: dict) -> str:
    """딕셔너리로 온 초안들을 하나의 글로 합친다."""
    parts = []
    for key, text in drafts.items():
        parts.append(f"[{LABELS[key]}]\n{text}")
    return "영업 메일 초안들:\n\n" + "\n\n---다음 초안---\n\n".join(parts)


pick_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "당신은 여러 영업 메일 초안 중 가장 좋은 것 하나를 고릅니다.\n"
     "당신이 이 메일을 받는 고객이라고 상상하고, 답장하고 싶은 메일을 고르세요.\n"
     "먼저 첫 줄에 '선택: 판매원N'만 적고, 줄을 바꾼 뒤 그 메일 전문을 그대로 출력하세요."),
    ("human", "{emails}"),
])

picker = pick_prompt | llm | parser
print("선택 체인 준비 완료")

선택 체인 준비 완료


### 왜 "선택: 판매원N"을 먼저 쓰게 했나

원본 노트북에서는 선택자가 메일 전문만 출력해서, **어느 초안이 뽑혔는지 알아내려고
앞 40자를 대조**해야 했습니다. 선택자가 문장을 조금 다듬으면 그 방법도 실패합니다.

**출력 형식을 정해 주면 후처리가 사라집니다.**
모델에게 "어떻게 답할지"를 지정하는 것은 후처리 코드를 줄이는 가장 값싼 방법입니다.

In [9]:
# 합치기 -> 선택. 전체가 하나의 체인이 된다.
select_chain = writers | RunnableLambda(join_drafts) | picker

best = select_chain.invoke({"request": "영업 메일 초안을 하나 써 주세요"})

print(best[:600])

선택: 판매원1
물론입니다. 아래는 **스나이퍼 APT**를 소개하는 **정직한 톤의 영업 메일 초안**입니다.

---

**제목:** 알려지지 않은 악성코드와 표적 공격 대응을 위한 스나이퍼 APT 소개드립니다

안녕하세요,  
주식회사 윈스테크넷 영업 담당자입니다.

최근 알려지지 않은 악성코드와 표적 공격은 기존의 단일 보안 솔루션만으로는 대응이 어려운 경우가 많습니다.  
이와 관련해 저희가 안내드리고자 하는 제품은 **스나이퍼 APT**입니다.

스나이퍼 APT는 **네트워크 단에서 알려지지 않은 악성코드와 표적 공격을 탐지하고 차단하는 APT 대응 솔루션**입니다.  
의심스러운 통신을 기반으로 위협을 확인하고 대응하는 데 도움을 드릴 수 있습니다.

다만, 솔직히 말씀드리면 스나이퍼 APT만으로 모든 보안 문제가 해결되는 것은 아닙니다.  
엔드포인트, 계정 보안, 취약점 관리, 사용자 교육 등은 별도의 보안 체계와 함께 운영되는 것이 바람직합니다.  
또한 위협 탐지 및 차단은 환경 구성과 정책에 따라 차이가 있을 수 있습니다.

관심 있으시면 제품 소개자료나 상세 설명을 드리겠습니다.  
필요하시면 현재 운영 중인 보안 환경에 맞춰 적용 가


In [10]:
print("첫 줄:", best.splitlines()[0])

첫 줄: 선택: 판매원1


> **판매원3은 뽑혔나요?**
>
> 대개 뽑히지 않습니다. "안 사면 망한다"는 압박은 보내는 쪽에서는 강력해 보이지만
> **받는 쪽에서는 답장할 이유가 되지 않습니다.**
> 선택자에게 "고객 입장이 되어 보라"고 시켰기 때문에 이 차이가 드러납니다.
>
> 몇 번 더 실행해서 매번 같은 판매원이 뽑히는지 확인해 보세요.
> 결과가 흔들린다면 선택 기준을 더 구체적으로 적어야 한다는 신호입니다.

## Step 6. 도구 만들기

`@tool` 데코레이터를 붙이면 평범한 함수가 모델이 쓸 수 있는 도구가 됩니다.
OpenAI Agents SDK의 `@function_tool`과 같은 자리입니다.

### `parse_docstring=True`를 꼭 붙이세요

이것이 없으면 **매개변수 설명이 모델에게 전달되지 않습니다.**
독스트링에 `Args:`를 정성껏 써 놓아도 무시됩니다.

아래 셀에서 두 경우를 나란히 비교합니다.

In [12]:
@tool
def tool_without_parse(subject: str, text_body: str) -> str:
    """영업 메일을 발송한다.

    Args:
        subject: 메일 제목
        text_body: 메일 본문 (일반 텍스트)
    """
    return send_message(subject, text_body)


@tool(parse_docstring=True)
def send_email_tool(subject: str, text_body: str) -> str:
    """영업 메일을 잠재 고객에게 발송한다.

    Args:
        subject: 메일 제목
        text_body: 메일 본문 (일반 텍스트)
    """
    return send_message(subject, text_body)


import json as _json
print("parse_docstring 없이:")
print(_json.dumps(tool_without_parse.args, indent=2, ensure_ascii=False))
print()
print("parse_docstring=True:")
print(_json.dumps(send_email_tool.args, indent=2, ensure_ascii=False))

parse_docstring 없이:
{
  "subject": {
    "title": "Subject",
    "type": "string"
  },
  "text_body": {
    "title": "Text Body",
    "type": "string"
  }
}

parse_docstring=True:
{
  "subject": {
    "description": "메일 제목",
    "title": "Subject",
    "type": "string"
  },
  "text_body": {
    "description": "메일 본문 (일반 텍스트)",
    "title": "Text Body",
    "type": "string"
  }
}


`description`이 있고 없고의 차이입니다.
매개변수가 두 개라 지금은 티가 덜 나지만, 도구가 늘고 매개변수가 복잡해지면
**설명이 없는 도구는 모델이 엉뚱한 값을 채워 넣습니다.**

## Step 7. 도구 실행 — 여기가 가장 큰 차이입니다

`bind_tools`로 모델에게 도구를 알려 줍니다.

```python
llm.bind_tools([send_email_tool])
```

**그런데 모델은 도구를 실행하지 않습니다.** "이 도구를 이런 인자로 불러 주세요"라고
**요청만** 합니다. 실제 실행은 **우리가** 해야 합니다.

| | OpenAI Agents SDK | LangChain |
|---|---|---|
| 도구 선택 | 모델 | 모델 |
| **도구 실행** | **SDK가 대신** | **우리가 직접** |
| 결과를 모델에 되돌리기 | SDK가 대신 | 우리가 직접 |
| 반복 | SDK가 알아서 | 우리가 루프 작성 |

먼저 모델이 무엇을 돌려주는지 직접 봅니다.

In [13]:
sender_llm = llm.bind_tools([send_email_tool], tool_choice="send_email_tool")

send_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "당신은 여러 영업 메일 초안 중 가장 좋은 것 하나를 고릅니다.\n"
     "고객 입장에서 답장하고 싶은 메일을 고르세요.\n"
     "고른 다음에는 반드시 도구를 사용해 그 메일을 발송하세요."),
    ("human", "{emails}"),
])

# parser를 붙이지 않는다. 도구 호출 정보를 봐야 하므로 AIMessage 그대로 받는다.
emails_text = join_drafts(drafts)
ai_message = (send_prompt | sender_llm).invoke({"emails": emails_text})

print("본문(content):", repr(ai_message.content)[:80])
print()
print("도구 호출 요청:")
for call in ai_message.tool_calls:
    print("  이름:", call["name"])
    print("  제목:", call["args"]["subject"])
    print("  본문 길이:", len(call["args"]["text_body"]), "자")

본문(content): ''

도구 호출 요청:
  이름: send_email_tool
  제목: 네트워크 단에서 APT 위협 탐지·차단을 지원하는 ‘스나이퍼 APT’ 안내
  본문 길이: 584 자


**`content`가 비어 있는 점을 보세요.** 모델은 할 말 대신 "도구를 불러 달라"는 요청을 보냈습니다.
이 시점에 메일은 **아직 나가지 않았습니다.**

`tool_choice="send_email_tool"`은 OpenAI Agents SDK의 `ModelSettings(tool_choice="required")`와
같은 역할입니다. 없으면 모델이 "이 메일이 가장 좋습니다"라고 **말만 하고 끝내기도** 합니다.

이제 우리가 직접 실행합니다.

In [14]:
for call in ai_message.tool_calls:
    result = send_email_tool.invoke(call["args"])
    print("도구 반환값:", result)

[미리보기] 실제로 보내지 않았습니다
제목: 네트워크 단에서 APT 위협 탐지·차단을 지원하는 ‘스나이퍼 APT’ 안내
------------------------------------------------------------------
안녕하세요,
주식회사 윈스테크넷입니다.

최근에는 알려진 악성코드뿐 아니라, 기존 시그니처 기반 탐지만으로는 확인이 어려운 표적 공격과 지능형 지속 위협(APT)에 대한 대응이 중요해지고 있습니다.
저희 ‘스나이퍼 APT’는 이런 위협을 네트워크 단에서 탐지하고 차단하는 데 초점을 둔 솔루션입니다.

스나이퍼 APT는 다음과 같은 상황에서 활용될 수 있습니다.

- 알려지지 않은 악성코드의 이상 징후 탐지
- 표적 공격과 APT 트래픽 식별
- 네트워크 구간에서의 차단 및 대응 지원

다만 솔직하게 말씀드리면,
스나이퍼 APT가 모든 보안 문제를 단독으로 해결하는 것은 아닙니다.
예를 들어 엔드포인트 자체의 취약점, 계정 탈취, 내부 운영 절차상의 문제까지 모두 대신해 주지는 않습니다. 따라서 기존 보안 장비, 보안 정책, 사용자 관리와 함께 운영하실 때 더 적절합니다.

관심이 있으시다면, 귀사의 환경에 맞춰
- 적용 가능 여부
- 운영 방식
- 도입 시 확인해야 할 사항
을 간단히 안내드리겠습니다.

검토해 주셔서 감사합니다.
필요하시면 짧은 소개 자료 형태로도 다시 정리해 드리겠습니다.

감사합니다.
주식회사 윈스테크넷 드림
도구 반환값: 미리보기로 출력했습니다.


> 이 두 셀이 LangChain을 이해하는 핵심입니다.
> **모델의 "요청"과 실제 "실행" 사이에 우리 코드가 있습니다.**
>
> 번거로워 보이지만 장점이 있습니다. 실행 직전에 **검사하고, 기록하고, 막을 수 있습니다.**
> 메일 발송처럼 되돌릴 수 없는 동작에서는 이 틈이 오히려 안전장치가 됩니다.
>
> ```python
> for call in ai_message.tool_calls:
>     if len(call["args"]["text_body"]) > 5000:      # 우리가 끼워 넣는 검사
>         print("본문이 너무 깁니다. 발송을 취소합니다.")
>         continue
>     send_email_tool.invoke(call["args"])
> ```
>
> OpenAI Agents SDK에서는 SDK가 알아서 실행해 버리므로, 이런 검사를 넣으려면
> 도구 함수 **안에** 써야 합니다.

## Step 8. 전체를 하나의 체인으로

지금까지 만든 것을 이어 붙입니다.

```
writers  |  join_drafts  |  send_prompt | sender_llm  |  run_tools
 (병렬)      (합치기)          (고르기)                  (실행)
```

`run_tools`도 `RunnableLambda`로 감싸면 **전체가 하나의 `Runnable`** 이 됩니다.
그러면 이 파이프라인 자체를 다른 곳에서 부품으로 쓸 수 있습니다.

In [15]:
def run_tools(ai_message) -> str:
    """모델이 요청한 도구를 실제로 실행한다."""
    if not ai_message.tool_calls:
        return "도구 호출이 없었습니다: " + (ai_message.content or "")
    results = [send_email_tool.invoke(call["args"]) for call in ai_message.tool_calls]
    return " / ".join(results)


full_chain = (
    writers
    | RunnableLambda(join_drafts)
    | send_prompt
    | sender_llm
    | RunnableLambda(run_tools)
)

result = full_chain.invoke({"request": "영업 메일 초안을 하나 써 주세요"})
print("\n파이프라인 결과:", result)

[미리보기] 실제로 보내지 않았습니다
제목: 알려지지 않은 악성코드·표적 공격 대응을 위한 스나이퍼 APT 소개
------------------------------------------------------------------
안녕하세요, 담당자님.

주식회사 윈스테크넷 영업 담당자입니다.
지능형 지속 위협(APT) 대응을 검토하시는 분들께 저희 스나이퍼 APT를 간단히 소개드리고자 메일 드립니다.

스나이퍼 APT는 네트워크 단에서 알려지지 않은 악성코드와 표적 공격을 탐지하고 차단하는 솔루션입니다.
이미 알려진 위협만이 아니라, 정상 트래픽처럼 보이는 공격 징후도 분석 대상에 포함해 대응을 지원합니다.

다만 솔직히 말씀드리면, 이 제품만으로 모든 보안 문제를 해결할 수는 없습니다.
예를 들어 단말 자체의 취약점 관리, 계정 보안, 내부 사용자 행위 통제 같은 영역은 별도의 보안 체계가 함께 필요합니다.
따라서 스나이퍼 APT는 기존 보안 장비와 함께 운영할 때 더 효과적으로 활용될 수 있습니다.

관심이 있으시다면 제품 소개 자료나 간단한 상담을 통해 귀사 환경에 맞는 적용 가능 여부를 함께 검토해 드리겠습니다.

읽어주셔서 감사합니다.
주식회사 윈스테크넷
영업 담당자 드림

파이프라인 결과: 미리보기로 출력했습니다.


### 체인의 구조 보기

`|`로 이어 붙인 것이 실제로 어떤 순서인지 확인할 수 있습니다.

In [16]:
for i, step in enumerate(full_chain.steps, 1):
    name = getattr(step, "name", None) or type(step).__name__
    print(f"  {i}. {name}")

  1. RunnableParallel
  2. join_drafts
  3. ChatPromptTemplate
  4. _ChatModelBinding
  5. run_tools


## Step 9. 정리 — OpenAI Agents SDK와 무엇이 달랐나

같은 일을 두 프레임워크로 만들어 봤습니다.

| | OpenAI Agents SDK | LangChain |
|---|---|---|
| 기본 단위 | `Agent` — 성격을 가진 주체 | `Runnable` — 입력→출력 부품 |
| 엮는 법 | 도구 / 핸드오프 | `\|` 연산자 (LCEL) |
| 병렬 | `asyncio.gather` (결과는 리스트) | `RunnableParallel` (결과는 딕셔너리) |
| 도구 실행 | **SDK가 대신 실행** | **우리가 직접 실행** |
| 여러 번 판단 | SDK가 루프를 돌림 | 우리가 루프를 짬 |
| 추적 | `with trace("이름")` | 환경변수로 자동 (LangSmith) |
| 모델 교체 | OpenAI 중심 | `ChatOpenAI`만 바꾸면 됨 |

### 어느 쪽을 쓰나

- **흐름이 정해져 있다** → LangChain. `|`로 이어 붙이면 읽기 쉽고 예측 가능합니다.
- **모델이 판단을 반복해야 한다** → Agents SDK. 루프를 대신 돌려 줍니다.
- **도구 실행 직전에 개입해야 한다** → LangChain. 그 틈이 우리 코드입니다.
- **모델을 바꿔 가며 비교한다** → LangChain. 그러라고 만든 구조입니다.

### 이 노트북에서 기억할 것

| | |
|---|---|
| LangChain은 연결 프레임워크 | 에이전트가 아니라 파이프라인입니다 |
| `StrOutputParser`를 끝에 | 없으면 `AIMessage` 객체가 흘러다닙니다 |
| `parse_docstring=True` | 없으면 매개변수 설명이 모델에게 안 갑니다 |
| `bind_tools`는 실행하지 않음 | 요청만 돌려줍니다. 실행은 우리 몫 |
| 출력 형식을 정해 주기 | 후처리 코드가 통째로 사라집니다 |
| LangSmith 자동 추적 | 편하지만, 끄는 것을 잊으면 전부 전송됩니다 |

### 더 해볼 것

1. `chain1`의 `llm`만 다른 모델로 바꿔 보세요. 나머지 코드는 그대로 돌아갑니다.
2. `tool_choice`를 빼고 모델이 도구를 안 부르는지 확인해 보세요.
3. `run_tools`에 본문 길이 검사를 넣어 발송을 막아 보세요.
4. `writers`에 판매원을 한 명 더 추가해 보세요 (`RunnableParallel`에 한 줄).

> 다음: **`langgraph_agent_code.ipynb`** — 같은 일을 그래프로 만듭니다.
> LangChain의 `|`는 **일직선**이라 되돌아가거나 갈라지기 어렵습니다.
> "검수에 떨어지면 다시 쓰게 한다" 같은 흐름은 그래프가 필요합니다.